In [2]:
import math

def vbeam(d1, d2, d3, b, L):
    """
    Calculates the properties of a composite sandwich beam.
    """
    # Material Constants (Density: kg/m^3, Young's Modulus: Pa, Cost: $/m^3)
    rho1, E1, c1 = 100, 1.6e9, 500
    rho2, E2, c2 = 2770, 70e9, 1500
    rho3, E3, c3 = 7780, 200e9, 800
    
    # Intermediate Calculations for Bending Stiffness (EI) and Mass per unit length (mu)
    EI = (2 * b / 3) * (E1 * d1**3 + E2 * (d2**3 - d1**3) + E3 * (d3**3 - d2**3))
    mu = 2 * b * (rho1 * d1 + rho2 * (d2 - d1) + rho3 * (d3 - d2))
    
    # Objective Functions and Constraints
    f0 = (math.pi / (2 * L**2)) * math.sqrt(EI / mu)l
    c = 2 * b * L * (c1 * d1 + c2 * (d2 - d1) + c3 * (d3 - d2))
    M = mu * L
    d21 = d2 - d1
    d32 = d3 - d2
    
    return f0, c, M, d21, d32

# Step 3: Test Values
print("Please enter your design parameters.")

# Wrapping inputs in float() to convert the typed text into decimal numbers
d1 = float(input("Enter d1: "))
d2 = float(input("Enter d2: "))
d3 = float(input("Enter d3: "))
b = float(input("Enter width b: "))
L = float(input("Enter length L: "))

f0, c, M, d21, d32 = vbeam(d1, d2, d3, b, L)

# Displaying the results formatted to match the test case expectations
print(f"Fundamental Frequency (f0): {f0:.4f} Hz")
print(f"Cost (c): ${c:,.0f}")
print(f"Mass (M): {M:,.0f} kg")
print(f"Layer 2 width (d21): {d21:.2f} m")
print(f"Layer 3 width (d32): {d32:.2f} m")

Please enter your design parameters.


Enter d1:  0.3
Enter d2:  0.35
Enter d3:  0.4
Enter width b:  5
Enter length L:  5


Fundamental Frequency (f0): 112.6849 Hz
Cost (c): $13,250
Mass (M): 27,875 kg
Layer 2 width (d21): 0.05 m
Layer 3 width (d32): 0.05 m


In [ ]:
import numpy as np
import math
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize
from pymoo.visualization.scatter import Scatter

class SandwichBeamProblem(ElementwiseProblem):
    def __init__(self):
        # Define 5 variables, 2 objectives, 2 inequality constraints, and 2 equality constraints
        super().__init__(
            n_var=3,
            n_obj=2,
            n_ieq_constr=4,
            n_eq_constr=0,
            # Lower bounds for [d1, d2, b] from Table 6.3
            xl=np.array([0.01, 0.01, 0.3]), 
            # Upper bounds for [d1, d2, d3, b, L] from Table 6.3
            xu=np.array([0.30, 0.35, 0.7])  
        )

    def _evaluate(self, x, out, *args, **kwargs):
        d1, d2, b = x

        d3 = 0.345
        
        # Material Constants
        rho1, E1, c1 = 100, 1.6e9, 500
        rho2, E2, c2 = 2770, 70e9, 1500
        rho3, E3, c3 = 7780, 200e9, 800
        
        # Bending Stiffness and Mass per unit length
        EI = (2 * b / 3) * (E1 * d1**3 + E2 * (d2**3 - d1**3) + E3 * (d3**3 - d2**3))
        mu = 2 * b * (rho1 * d1 + rho2 * (d2 - d1) + rho3 * (d3 - d2))
        L = 1845.0 / mu
        
        # Calculate properties
       # Protect against negative values in the square root during random exploration
        ratio = EI / mu
        if ratio > 0:
            f0 = (math.pi / (2 * L**2)) * math.sqrt(ratio)
        else:
            f0 = 0.0  # Assign a terrible fitness to invalid geometries
            
        c = 2 * b * L * (c1 * d1 + c2 * (d2 - d1) + c3 * (d3 - d2))
                
        # 1. Objectives (Minimize)
        # We negate f0 to frame the maximization as a minimization problem
        out["F"] = [-f0, c]
        
        # 2. Inequality Constraints (Must be <= 0)
        out["G"] = [
            0.001 - (d2 - d1),  # d21 >= 0.001
            0.001 - (d3 - d2),   # d32 >= 0.001
            3.0 - L,
            L - 6.0
        ]
        
        # Initialize the problem
problem = SandwichBeamProblem()

# Configure the NSGA-II algorithm to generate 100 solutions
algorithm = NSGA2(pop_size=500)

# Run the optimization
res = minimize(
    problem,
    algorithm,
    ("n_gen", 1000),
    #seed=1,
    verbose=True
)

# Extract the results: reverse the negation to display the true frequency
frequencies = -res.F[:, 0]
costs = res.F[:, 1]

plot = Scatter(title="Pareto Frontier: Frequency vs. Cost", labels=["Fundamental Frequency (Hz)", "Cost ($)"])
plot.add(np.column_stack((frequencies, costs)))
plot.show()

frequencies = -res.F[:, 0]
costs = res.F[:, 1]
designs = res.X

valid_indices = np.where(costs <= 600.0)[0]

if len(valid_indices) > 0:
    # Isolate the valid frequencies, costs, and variables
    valid_freqs = frequencies[valid_indices]
    valid_costs = costs[valid_indices]
    valid_designs = designs[valid_indices]

    # 2. Find the index of the maximum frequency within this valid subset
    best_idx = np.argmax(valid_freqs)

    # 3. Extract the best objectives
    best_f0 = valid_freqs[best_idx]
    best_c = valid_costs[best_idx]

    # 4. Extract the primary search variables for this specific design
    best_d1, best_d2, best_b = valid_designs[best_idx]
    
    # 5. Recalculate the derived variables (L, d3, d21, d32, M)
    d3 = 0.345
    rho1, rho2, rho3 = 100, 2770, 7780
    
    mu = 2 * best_b * (rho1 * best_d1 + rho2 * (best_d2 - best_d1) + rho3 * (d3 - best_d2))
    best_L = 1845.0 / mu
    best_d21 = best_d2 - best_d1
    best_d32 = d3 - best_d2
    best_M = mu * best_L

    print(f"Max Achievable Frequency (f0): {best_f0:.4f} Hz")
    print(f"Actual Cost (c): ${best_c:,.2f}")
    print("\nRequired Design Variables:")
    print(f"d1 (Material 1 depth): {best_d1:.4f} m")
    print(f"d2 (Material 2 depth): {best_d2:.4f} m")
    print(f"d3 (Total depth):      {d3:.4f} m")
    print(f"b  (Beam width):       {best_b:.4f} m")
    print(f"L  (Beam length):      {best_L:.4f} m")
    print("\nLayer Dimensions:")
    print(f"d21 (Layer 2 width):   {best_d21:.4f} m")
    print(f"d32 (Layer 3 width):   {best_d32:.4f} m")
    print(f"Total Mass (M):        {best_M:,.0f} kg")

else:
    print("The algorithm could not find any feasible solutions under $600.")
    print("Try increasing the population size or number of generations.")

n_gen  |  n_eval  | n_nds  |     cv_min    |     cv_avg    |      eps      |   indicator  
     1 |      500 |      4 |  0.000000E+00 |  1.4669013544 |             - |             -
     2 |     1000 |      7 |  0.000000E+00 |  0.5979144202 |  0.0419686552 |             f
     3 |     1500 |      7 |  0.000000E+00 |  0.0698263773 |  0.0209823661 |         ideal
     4 |     2000 |     10 |  0.000000E+00 |  0.000000E+00 |  0.0657812352 |         ideal
     5 |     2500 |     13 |  0.000000E+00 |  0.000000E+00 |  0.0210088844 |             f
     6 |     3000 |     14 |  0.000000E+00 |  0.000000E+00 |  0.0145308872 |         ideal
     7 |     3500 |      8 |  0.000000E+00 |  0.000000E+00 |  0.1101148371 |         ideal
     8 |     4000 |     10 |  0.000000E+00 |  0.000000E+00 |  0.0058282568 |         ideal
     9 |     4500 |     12 |  0.000000E+00 |  0.000000E+00 |  0.0772053211 |         ideal
    10 |     5000 |     14 |  0.000000E+00 |  0.000000E+00 |  0.0175972396 |             f